# 🚀 Fine-Tuning & Transfer Learning com Unsloth no Google Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yanchagas04/personal-assistant/blob/feat/transfer_learning/notebooks/transfer_learning_unsloth.ipynb)

Este notebook realiza **Fine-Tuning real com ajuste de pesos (LoRA / QLoRA)** utilizando a biblioteca **Unsloth**, a ferramenta mais rápida e eficiente em consumo de memória para treinar modelos da família **LLaMA 3 / LLaMA 3.2**.

> ⚡ **Vantagens do Unsloth no Colab (GPU T4 Gratuita):**
> - Treinamento **2x a 5x mais rápido** com até 80% menos uso de VRAM.
> - Ajusta os pesos neurais do modelo para absorver com fidelidade o vocabulário, gírias e tom de Yan Chagas.
> - Exporta o modelo treinado diretamente para **GGUF**, permitindo importá-lo no seu **Ollama local**!

---

### 📌 Roteiro do Notebook:
1. **Instalação do Unsloth e Dependências** (otimizado para Colab GPU).
2. **Carregamento do Modelo Base LLaMA 3.2 em 4-bit** (`unsloth/Llama-3.2-3B-Instruct`).
3. **Configuração dos Adaptadores LoRA** (treinando apenas 1-2% dos parâmetros).
4. **Carregamento e Formatação do Dataset** (`personal-answers.jsonl`).
5. **Treinamento com `SFTTrainer`** (~3 a 5 minutos na GPU T4).
6. **Inferência e Testes Comparativos** (validando o comportamento do clone).
7. **Exportação para GGUF** (`q4_k_m`).
8. **Salvar no Google Drive** (armazenamento permanente do arquivo GGUF).
9. **Como Usar no Ollama Local** (criação do Modelfile e execução).

## 1. Verificação de GPU e Instalação do Unsloth

In [1]:
# Valida se o ambiente do Colab possui GPU ativa (T4, V100 ou A100)
!nvidia-smi

# Instalação do Unsloth e dependências compatíveis com Torch e CUDA do Colab
!pip install --no-deps "xformers<0.0.28" "trl<0.9.0" peft accelerate bitsandbytes
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q datasets python-dotenv

Fri Sep 25 15:31:50 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   35C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Carregamento do Modelo Base em 4-bit

Utilizamos o **`Llama-3.2-3B-Instruct`** pré-quantizado em 4 bits da Unsloth (leve, rápido e excelente para diálogos em português).

In [2]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None  # None para detecção automática (Float16 ou Bfloat16)
load_in_4bit = True

# Modelo recomendado para clone digital no Colab
MODEL_NAME = "unsloth/Llama-3.2-3B-Instruct"

print(f"Carregando modelo {MODEL_NAME}...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)
print("✅ Modelo base e Tokenizer carregados com sucesso!")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Carregando modelo unsloth/Llama-3.2-3B-Instruct...
==((====))==  Unsloth 2026.9.11: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3.2-3b-instruct-unsloth-bnb-4bit as a legacy tokenizer.


✅ Modelo base e Tokenizer carregados com sucesso!


## 3. Configuração dos Adaptadores LoRA / QLoRA

Configuramos o LoRA para injetar matrizes de rank baixo nas camadas de atenção e MLP, permitindo que o modelo aprenda o novo estilo sem perder seu conhecimento prévio.

In [3]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,                # Rank do LoRA (16 ou 32 são ideais)
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0,      # Otimizado para 0 no Unsloth
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

print("✅ Adaptadores LoRA aplicados!")

Unsloth 2026.9.11 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


✅ Adaptadores LoRA aplicados!


## 4. Carregamento e Formatação do Dataset de Respostas Pessoais

Carregamos o arquivo `personal-answers.jsonl` (enviado diretamente para o ambiente do Colab) contendo as respostas reais e autênticas do Yan Chagas, aplicando o template oficial de chat do LLaMA-3.

In [7]:
import json
from pathlib import Path
from datasets import Dataset
from unsloth.chat_templates import get_chat_template

# Dataset enviado diretamente para o ambiente do Colab
data_path = Path("personal-answers.jsonl")

# Fallback caso tenha subido como chat_dataset_sample25.jsonl
if not data_path.exists() and Path("chat_dataset_sample25.jsonl").exists():
    data_path = Path("chat_dataset_sample25.jsonl")

with open(data_path, "r", encoding="utf-8") as f:
    raw_dialogues = [json.loads(line) for line in f if line.strip()]

print(f"Total de conversas carregadas ({data_path.name}): {len(raw_dialogues)}")

# Aplicar Chat Template do Llama-3.1 / Llama-3.2
tokenizer = get_chat_template(
    tokenizer,
    chat_template="llama-3.1",
)

def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False) for convo in convos]
    return {"text": texts}

hf_dataset = Dataset.from_list(raw_dialogues)
hf_dataset = hf_dataset.map(formatting_prompts_func, batched=True)

print("\n--- Exemplo formatado para treino ---")
print(hf_dataset[0]["text"][:400] + "...")


Total de conversas carregadas (personal-answers.jsonl): 48


Map:   0%|          | 0/48 [00:00<?, ? examples/s]


--- Exemplo formatado para treino ---
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 July 2024

Você é o clone digital do usuário. Responda diretamente, de forma concisa e mantendo o tom informal e autêntico.<|eot_id|><|start_header_id|>user<|end_header_id|>

E aí, bora sair hoje à noite?<|eot_id|><|start_header_id|>assistant<|end_header_id|>

Porra, negão, tô sem vont...


## 5. Treinamento Supervisionado com `SFTTrainer`

Iniciamos o ajuste fino dos adaptadores LoRA. Como temos 25 conversas de alta qualidade, o treino leva **menos de 3 minutos** no Colab.

In [8]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=hf_dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=42,
        output_dir="outputs",
        report_to="none",
    ),
)

print("Iniciando treinamento...")
trainer_stats = trainer.train()
print("🎉 Treinamento concluído com sucesso!")

Unsloth: transformers renamed `push_to_hub_token` to `hub_token`. Forwarding your value to `hub_token` - update your code when convenient. If you also passed `hub_token` as None, that is its default here and cannot be distinguished from leaving it unset, so `push_to_hub_token` was used; drop `push_to_hub_token` to keep it.
Unsloth: We found double BOS tokens - we shall remove one automatically.


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/48 [00:00<?, ? examples/s]

Iniciando treinamento...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 48 | Num Epochs = 3 | Total steps = 18
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 24,313,856 of 3,237,063,680 (0.75% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!
/usr/local/lib/python3.13/dist-packages/unsloth/import_fixes.py:4217: UserWarning: 'has_cuda' is deprecated, please use 'torch.backends.cuda.is_built()'
  return original(name)
/usr/local/lib/python3.13/dist-packages/unsloth/import_fixes.py:4217: UserWarning: 'has_cudnn' is deprecated, please use 'torch.backends.cudnn.is_available()'
  return original(name)
/usr/local/lib/python3.13/dist-packages/unsloth/import_fixes.py:4217: UserWarning: 'has_mps' is deprecated, please use 'torch.backends.mps.is_built()'
  return original(name)
/usr/local/lib/python3.13/dist-packages/

Step,Training Loss
1,4.145125
2,4.035627
3,3.993017
4,3.578160
5,3.230279
6,2.914902
7,2.430840
8,2.001761
9,1.847835
10,1.711931


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-18/tokenizer_config.json.


🎉 Treinamento concluído com sucesso!


## 6. Testes de Inferência do Clone Treinado

In [9]:
# Prepara modelo para inferência acelerada
FastLanguageModel.for_inference(model)

test_messages = [
    {"role": "system", "content": "Você é o clone digital de Yan Chagas. Responde direto ao ponto e de forma autêntica."},
    {"role": "user", "content": "Bora almoçar onde hoje?"}
]

inputs = tokenizer.apply_chat_template(
    test_messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt"
).to("cuda")

outputs = model.generate(input_ids=inputs, max_new_tokens=64, use_cache=True, temperature=0.7)
decoded = tokenizer.batch_decode(outputs)
print("🤖 Resposta gerada:")
print(decoded[0].split("<|start_header_id|>assistant<|end_header_id|>")[-1].replace("<|eot_id|>", "").strip())

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Both `max_new_tokens` (=64) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🤖 Resposta gerada:
Não sei, mas posso ajudar a decidir.


## 7. Exportação do Modelo para GGUF (Pronto para Ollama!)

O Unsloth permite exportar o modelo treinado diretamente para o formato **GGUF** (quantização `q4_k_m`), que é o formato nativo do **Ollama**!

In [10]:
# Salva e quantiza diretamente em formato GGUF
GGUF_DIR = "yan_clone_gguf"

print("Exportando modelo para GGUF (quantização q4_k_m)... aguarde alguns instantes...")
model.save_pretrained_gguf(GGUF_DIR, tokenizer, quantization_method="q4_k_m")
print(f"✅ Modelo GGUF exportado em: {GGUF_DIR}")

# Listar arquivos exportados
!ls -lh {GGUF_DIR}

Unsloth: ##### The current model auto adds a BOS token.
Unsloth: ##### Your chat template has a BOS token. We shall remove it temporarily.


Exportando modelo para GGUF (quantização q4_k_m)... aguarde alguns instantes...
Unsloth: Merging model weights to 16-bit format...


config.json:   0%|          | 0.00/890 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/20.9k [00:00<?, ?B/s]

Unsloth: Restored added_tokens_decoder metadata in yan_clone_gguf/tokenizer_config.json.


...

Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [01:18<00:00, 39.35s/it]


Unsloth: Merge process complete. Saved to `/content/yan_clone_gguf`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Installing prebuilt llama.cpp b11160-mix-a6922cc (app-b11160-mix-a6922cc-linux-x64-cpu.tar.gz) - skipping compilation.
Unsloth: Preparing converter script...


Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['yan_clone_gguf_gguf/llama-3.2-3b-instruct.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q4_k_m. This might take 10 minutes...


Unsloth: ##### The current model auto adds a BOS token.
Unsloth: ##### We removed it in GGUF's chat template for you.


Unsloth: Model files cleanup...
Unsloth: All GGUF conversions completed successfully!
Generated files: ['yan_clone_gguf_gguf/llama-3.2-3b-instruct.Q4_K_M.gguf']
Unsloth: example usage for text only LLMs: /root/.unsloth/llama.cpp/llama-cli --model yan_clone_gguf_gguf/llama-3.2-3b-instruct.Q4_K_M.gguf -p "why is the sky blue?"
Unsloth: Saved Ollama Modelfile to yan_clone_gguf_gguf/Modelfile
Unsloth: convert model to ollama format by running - ollama create model_name -f yan_clone_gguf_gguf/Modelfile
✅ Modelo GGUF exportado em: yan_clone_gguf
total 6.1G
-rw-r--r-- 1 root root 4.5K Sep 25 15:56 chat_template.jinja
-rw-r--r-- 1 root root  940 Sep 25 15:56 config.json
-rw-r--r-- 1 root root  233 Sep 25 15:56 generation_config.json
-rw-r--r-- 1 root root 4.7G Sep 25 15:57 model-00001-of-00002.safetensors
-rw-r--r-- 1 root root 1.4G Sep 25 15:58 model-00002-of-00002.safetensors
-rw-r--r-- 1 root root  21K Sep 25 15:56 model.safetensors.index.json
-rw-r--r-- 1 root root  55K Sep 25 15:56 tokeni

## 8. Salvar no Google Drive (Permanente)

Como a máquina virtual do Google Colab é temporária (o disco e arquivos locais são descartados ao fechar a sessão), execute a célula abaixo para **montar o seu Google Drive** e salvar o arquivo `.gguf` gerado em definitivo na nuvem.

In [11]:
# Monta o Google Drive e copia o modelo GGUF exportado de forma permanente
import shutil
from pathlib import Path
from google.colab import drive

# 1. Conecta ao Google Drive pessoal
drive.mount('/content/drive')

# 2. Cria diretório de destino no seu Google Drive (ex: 'yan_clone_gguf')
drive_output_dir = Path('/content/drive/MyDrive/yan_clone_gguf')
drive_output_dir.mkdir(parents=True, exist_ok=True)

# 3. Copia os arquivos .gguf gerados para o Drive
print(f"Copiando modelo GGUF para o Google Drive ({drive_output_dir})...")
gguf_files = list(Path("yan_clone_gguf").glob("*.gguf"))

if gguf_files:
    for gguf_file in gguf_files:
        dest_file = drive_output_dir / gguf_file.name
        shutil.copy(gguf_file, dest_file)
        size_gb = dest_file.stat().st_size / (1024**3)
        print(f"✅ Arquivo salvo no Drive com sucesso: {dest_file.name} ({size_gb:.2f} GB)")
    print(f"\n🎉 Tudo pronto! Modelo salvo permanentemente em: {drive_output_dir}")
else:
    print("⚠️ Nenhum arquivo .gguf encontrado em 'yan_clone_gguf'. Verifique a execução da célula 7.")

# Dica: Você também pode fazer download direto pelo navegador descomentando abaixo:
# from google.colab import files
# if gguf_files:
#     files.download(str(gguf_files[0]))

Mounted at /content/drive
Copiando modelo GGUF para o Google Drive (/content/drive/MyDrive/yan_clone_gguf)...
⚠️ Nenhum arquivo .gguf encontrado em 'yan_clone_gguf'. Verifique a execução da célula 7.


## 9. Como Usar o Modelo Treinado no seu Ollama Local

1. Baixe o arquivo `.gguf` gerado (ex: `yan_clone_gguf/Llama-3.2-3B-Instruct-Q4_K_M.gguf`).
2. No seu computador, crie um arquivo `Modelfile` apontando para o arquivo baixado:
   ```dockerfile
   FROM ./Llama-3.2-3B-Instruct-Q4_K_M.gguf
   PARAMETER temperature 0.7
   SYSTEM "Você é o clone digital de Yan Chagas. Responde direto ao ponto e de forma autêntica."
   ```
3. No terminal do seu computador, crie e rode o modelo:
   ```bash
   ollama create yan-clone-treinado -f Modelfile
   ollama run yan-clone-treinado
   ```